In [2]:
import pandas as pd
import numpy as np
import os

# Chỉ định chính xác đường dẫn đến thư mục chứa dữ liệu gốc của nhóm
DATA_DIR = "../../data/raw" 
file_path = os.path.join(DATA_DIR, 'data_public.csv')

# Nạp dữ liệu gốc
df = pd.read_csv(file_path)
print(f"Dữ liệu ban đầu: {df.shape[0]} dòng")

# 1. Xóa trùng lặp tuyệt đối (trùng Title, Price, Area, Location)
df = df.drop_duplicates(subset=['Title', 'Price', 'Area', 'Location'], keep='last')

# 2. Xóa các dòng mất thông tin cốt lõi (Không có giá hoặc không có diện tích)
df = df.dropna(subset=['Price', 'Area'])

print(f"Sau khi xóa rác cơ học: {df.shape[0]} dòng")

Dữ liệu ban đầu: 51304 dòng
Sau khi xóa rác cơ học: 48827 dòng


In [3]:
import re
import numpy as np

print(f"Số dòng trước khi cứu hộ: {df.shape[0]}")
print(f"Số dòng bị thiếu Giá: {df['Price'].isna().sum()}, thiếu Diện tích: {df['Area'].isna().sum()}")

# Gom Text để quét
text_corpus = (df['Title'].fillna('') + " " + df['Description'].fillna('')).str.lower()

# ---------------------------------------------------------
# 1. CỨU HỘ DIỆN TÍCH (Tìm các cụm từ như: 50m2, 50.5 m2, 50 mét vuông)
# ---------------------------------------------------------
# Biểu thức Regex lấy số đứng trước chữ "m2" hoặc "mét vuông"
extracted_area = text_corpus.str.extract(r'(\d+[\.,]?\d*)\s*(?:m2|m²|mét vuông|m\^2)')[0]
# Đổi dấu phẩy thành dấu chấm (VD: 50,5 -> 50.5) và ép kiểu số
extracted_area = extracted_area.str.replace(',', '.').astype(float)

# Lấp đầy cột Area hiện tại bằng dữ liệu vừa cào được (chỉ lấp vào chỗ bị NaN)
df['Area'] = df['Area'].fillna(extracted_area)

# ---------------------------------------------------------
# 2. CỨU HỘ GIÁ TIỀN (Tìm các cụm: 3.5 tỷ, 800 triệu)
# ---------------------------------------------------------
# A. Lấy giá theo TỶ (Nhân 1000 để đổi ra đơn vị Triệu VNĐ)
price_ty = text_corpus.str.extract(r'(\d+[\.,]?\d*)\s*(?:tỷ|tỉ)')[0]
price_ty = price_ty.str.replace(',', '.').astype(float) * 1000

# B. Lấy giá theo TRIỆU
price_trieu = text_corpus.str.extract(r'(\d+[\.,]?\d*)\s*(?:triệu|tr)')[0]
price_trieu = price_trieu.str.replace(',', '.').astype(float)

# Gom 2 loại giá lại (ưu tiên giá Tỷ, nếu không có thì lấy giá Triệu)
extracted_price = price_ty.fillna(price_trieu)

# Lấp đầy cột Price hiện tại (chỉ lấp vào chỗ bị NaN)
df['Price'] = df['Price'].fillna(extracted_price)

print(f"Sau khi cứu hộ - Số dòng thiếu Giá: {df['Price'].isna().sum()}, thiếu Diện tích: {df['Area'].isna().sum()}")

# ---------------------------------------------------------
# 3. LÚC NÀY MỚI XÓA NHỮNG DÒNG VẪN KHÔNG TÌM THẤY
# ---------------------------------------------------------
df = df.dropna(subset=['Price', 'Area'])
print(f"Số dòng giữ lại thành công để đi tiếp: {df.shape[0]}")

Số dòng trước khi cứu hộ: 48827
Số dòng bị thiếu Giá: 0, thiếu Diện tích: 0
Sau khi cứu hộ - Số dòng thiếu Giá: 0, thiếu Diện tích: 0
Số dòng giữ lại thành công để đi tiếp: 48827


In [4]:
# Ép kiểu dữ liệu về số thực (float)
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
df['Area'] = pd.to_numeric(df['Area'], errors='coerce')

# Bắt buộc xóa các dòng không thể chuyển sang số
df = df.dropna(subset=['Price', 'Area'])

# LOGIC 1: Đơn vị là TRIỆU VNĐ. Nhà rẻ nhất TP.HCM không thể dưới 500 triệu. Đắt nhất tạm để 5000 tỷ.
df = df[(df['Price'] >= 500) & (df['Price'] <= 5000000)]

# LOGIC 2: Diện tích không thể dưới 10m2 (không đủ chuẩn cấp sổ) và khó vượt quá 10,000m2 (nếu là nhà phố/chung cư)
df = df[(df['Area'] >= 10) & (df['Area'] <= 10000)]

# LOGIC 3: Loại bỏ các tin đăng "cho thuê" bị cào nhầm vào (Giá thường quá rẻ so với diện tích)
# VD: Nhà 100m2 mà giá 15 triệu -> Đây là giá thuê, không phải giá bán
df['Price_per_m2'] = df['Price'] / df['Area']
df = df[df['Price_per_m2'] >= 10] # Giá/m2 ít nhất phải 10 triệu/m2 đối với mua bán

print(f"Sau khi lọc Logic BĐS: {df.shape[0]} dòng (Đây là tập Data Sạch Thật Sự)")

Sau khi lọc Logic BĐS: 40319 dòng (Đây là tập Data Sạch Thật Sự)


In [5]:
text_corpus = (df['Title'].fillna('') + " " + df['Description'].fillna('')).str.lower()

df['Is_Mat_Tien'] = text_corpus.str.contains(r'mặt tiền|mt|mặt phố', regex=True).astype(int)
df['Is_Hem_Xe_Hoi'] = text_corpus.str.contains(r'hẻm xe hơi|hxh|hẻm ô tô|ô tô vào|xe hơi vào', regex=True).astype(int)
df['Has_So_Hong'] = text_corpus.str.contains(r'sổ hồng|sổ đỏ|pháp lý chuẩn|sẵn sổ|sổ riêng', regex=True).astype(int)

# Khai thác thêm: Xác định Căn hộ / Chung cư
df['Is_Chung_Cu'] = text_corpus.str.contains(r'chung cư|căn hộ|apartment', regex=True).astype(int)